# NeuralSpace Google Colab Bootstrap

Run the bootstrap cell below, paste your one-time claim code, then use the short helpers to make run lineage explicit: `ns.use(...)` records dataset/model inputs, `ns.metric(...)` or `ns.log_metrics(...)` records telemetry, and `ns.produce(...)` records output assets. The second cell shows the compact `start -> use -> train/log -> produce -> finish` flow.


In [ ]:
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse
import os
import socket
import requests


API_BASE_URL = ""  # @param {type:"string"}
DEFAULT_API_BASE_ENV = "PUBLIC_API_BASE_URL"


def resolve_api_base(api_base=None):
    resolved = (api_base or API_BASE_URL or os.environ.get(DEFAULT_API_BASE_ENV, "")).strip().rstrip("/")
    if not resolved:
        raise RuntimeError(
            "Set API_BASE_URL in this Colab cell, or set the PUBLIC_API_BASE_URL environment variable, "
            "to your current NeuralSpace API URL, for example https://<your-tunnel>.trycloudflare.com/api/v1."
        )
    parsed = urlparse(resolved)
    if parsed.path in ("", "/"):
        resolved = f"{resolved}/api/v1"
    return resolved


class NeuralSpaceAssetClient:
    def __init__(self, parent, kind):
        self.parent = parent
        self.kind = kind
        self.id_key = "dataset_id" if kind == "datasets" else "model_id"
        self.type_name = kind[:-1]

    @property
    def items(self):
        return self.parent.assets.get(self.kind, [])

    def list(self, refresh=False):
        if refresh:
            self.parent.refresh_assets()
        return self.items

    def get(self, asset_id):
        for item in self.items:
            if item.get(self.id_key) == asset_id:
                return item
        raise KeyError(f"{self.type_name} not attached to this workspace: {asset_id}")

    def pick(self, value=None, *, name=None, refresh=False):
        items = self.list(refresh=refresh)
        if value is None and name is None:
            if not items:
                raise RuntimeError(f"No {self.kind} attached to this workspace")
            return items[0]
        needle = value or name
        for item in items:
            if item.get(self.id_key) == needle or item.get("name") == needle:
                return item
        raise KeyError(f"{self.type_name} not attached to this workspace: {needle}")

    def download(self, asset=None, target_dir=None):
        item = self.pick(asset) if isinstance(asset, str) else asset
        if not item:
            item = self.pick()
        asset_id = item[self.id_key]
        signed_url = item.get("signed_url")
        if not signed_url:
            raise RuntimeError(f"No download URL available for {asset_id}")
        base_dir = Path(target_dir or f"/content/neuralspace/{self.kind}/{asset_id}")
        base_dir.mkdir(parents=True, exist_ok=True)
        filename = Path(urlparse(signed_url).path).name or f"{asset_id}.bin"
        output_path = base_dir / filename
        with requests.get(signed_url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with output_path.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        return str(output_path)

    def load(self, asset=None, target_dir=None):
        return self.download(asset, target_dir=target_dir)


class NeuralSpaceClient:
    def __init__(self, api_base, runtime_token, bootstrap):
        self.api_base = api_base.rstrip("/")
        self.runtime_token = runtime_token
        self.headers = {"Authorization": f"Bearer {runtime_token}"}
        self.session_id = bootstrap["session_id"]
        self.run_id = None
        self.assets = {
            "datasets": bootstrap.get("datasets", []),
            "models": bootstrap.get("models", []),
        }
        self.datasets = NeuralSpaceAssetClient(self, "datasets")
        self.models = NeuralSpaceAssetClient(self, "models")

    @classmethod
    def connect(cls, api_base=None):
        api_base = resolve_api_base(api_base)
        print(f"Using NeuralSpace API: {api_base}")
        if not (api_base.startswith("https://") or api_base.startswith("http://localhost") or api_base.startswith("http://127.0.0.1")):
            raise RuntimeError("Use HTTPS for remote NeuralSpace APIs")
        host = urlparse(api_base).hostname
        try:
            socket.gethostbyname(host)
        except OSError as exc:
            raise RuntimeError(
                f"Cannot resolve NeuralSpace API host: {host}. "
                "If you are using trycloudflare.com, restart the Cloudflare tunnel and paste the fresh public URL."
            ) from exc
        claim_code = getpass("Paste NeuralSpace claim code: ").strip()
        response = requests.post(f"{api_base}/colab/claims/exchange", json={"claim_code": claim_code}, timeout=30)
        response.raise_for_status()
        return cls(api_base, response.json()["runtime_token"], response.json())

    def request(self, method, path, **kwargs):
        response = requests.request(method, f"{self.api_base}{path}", headers=self.headers, timeout=kwargs.pop("timeout", 30), **kwargs)
        response.raise_for_status()
        return response.json()

    def heartbeat(self):
        return self.request("POST", "/colab/runtime/heartbeat")

    def refresh_assets(self):
        self.assets = self.request("GET", "/colab/runtime/assets")
        return self.assets

    def start_run(self, name="Colab run", inputs=None, outputs=None):
        payload = {"name": name}
        if inputs:
            payload["inputs"] = [self._asset_payload(item) for item in inputs]
        if outputs:
            payload["outputs"] = [self._asset_payload(item, role="output") for item in outputs]
        run = self.request("POST", "/colab/runtime/runs", json=payload)
        self.run_id = run["run_id"]
        return run

    def ensure_run(self, name="Colab run"):
        if not self.run_id:
            self.start_run(name)
        return self.run_id

    def use(self, asset, role="input"):
        run_id = self.ensure_run()
        payload = self._asset_payload(asset, role=role)
        return self.request("POST", f"/colab/runtime/runs/{run_id}/inputs", json=payload)

    def produce(self, asset, role="output"):
        run_id = self.ensure_run()
        payload = self._asset_payload(asset, role=role)
        return self.request("POST", f"/colab/runtime/runs/{run_id}/outputs", json=payload)

    def use_dataset(self, dataset, role="training_dataset"):
        return self.use(self._asset_payload(dataset, asset_type="dataset"), role=role)

    def use_model(self, model, role="input_model"):
        return self.use(self._asset_payload(model, asset_type="model"), role=role)

    def produce_model(self, model, role="output_model"):
        return self.produce(self._asset_payload(model, asset_type="model"), role=role)

    def metric(self, name, value, step=None):
        return self.log_metrics({name: value}, step=step)

    def log_metrics(self, values, step=None):
        run_id = self.ensure_run()
        payload = {"values": values}
        if step is not None:
            payload["step"] = step
        return self.request("POST", f"/colab/runtime/runs/{run_id}/metrics", json=payload)

    def log(self, message, level="INFO"):
        run_id = self.ensure_run()
        return self.request(
            "POST",
            f"/colab/runtime/runs/{run_id}/logs",
            json={"level": level, "message": message},
        )

    def finish_run(self, status="success"):
        run_id = self.ensure_run()
        return self.request("PATCH", f"/colab/runtime/runs/{run_id}", json={"status": status})

    def _asset_payload(self, asset, asset_type=None, role=None):
        if isinstance(asset, dict):
            payload = dict(asset)
            if "asset_type" not in payload:
                payload["asset_type"] = asset_type or self._infer_asset_type(payload)
            if "asset_id" not in payload:
                payload["asset_id"] = payload.get("dataset_id") or payload.get("model_id") or payload.get("id")
        else:
            if not asset_type:
                raise ValueError("Pass an asset object or provide asset_type for raw ids")
            payload = {"asset_type": asset_type, "asset_id": asset}
        if role:
            payload["role"] = role
        if not payload.get("asset_type") or not payload.get("asset_id"):
            raise ValueError("Asset payload must include asset_type and asset_id")
        return {
            "asset_type": payload["asset_type"],
            "asset_id": payload["asset_id"],
            "role": payload.get("role", role or "input"),
        }

    def _infer_asset_type(self, asset):
        if asset.get("dataset_id"):
            return "dataset"
        if asset.get("model_id"):
            return "model"
        return asset.get("type")


ns = NeuralSpaceClient.connect()
assets = ns.refresh_assets()
datasets = ns.datasets.list()
models = ns.models.list()
print(f"Connected to NeuralSpace session {ns.session_id}")
print(f"Datasets: {len(datasets)} | Models: {len(models)}")
assets


In [ ]:
# NeuralSpace short-helper template: reuse the `ns` connection from the bootstrap cell.
import time

RUN_NAME = "Colab lineage template"  # @param {type:"string"}
TOTAL_STEPS = 5  # @param {type:"integer"}
DATASET_ID = ""  # @param {type:"string"}
BASE_MODEL_ID = ""  # @param {type:"string"}
OUTPUT_MODEL_ID = ""  # @param {type:"string"}

train_dataset = None
if DATASET_ID.strip().upper() == "AUTO":
    train_dataset = ns.datasets.pick() if datasets else None
elif DATASET_ID.strip():
    train_dataset = ns.datasets.pick(DATASET_ID.strip())

base_model = None
if BASE_MODEL_ID.strip().upper() == "AUTO":
    base_model = ns.models.pick() if models else None
elif BASE_MODEL_ID.strip():
    base_model = ns.models.pick(BASE_MODEL_ID.strip())

run = ns.start_run(RUN_NAME)
print("Run created:", run["run_id"])

try:
    ns.log(f"Started {RUN_NAME} from Google Colab")

    if train_dataset:
        ns.use(train_dataset, role="training_dataset")
    if base_model:
        ns.use(base_model, role="base_model")

    for step in range(1, TOTAL_STEPS + 1):
        loss = round(1.0 / step, 4)
        accuracy = round(0.72 + step * 0.04, 4)
        learning_rate = round(0.001 * (0.9 ** (step - 1)), 6)

        ns.log_metrics(
            {
                "loss": loss,
                "accuracy": accuracy,
                "learning_rate": learning_rate,
            },
            step=step,
        )
        ns.log(f"Step {step}/{TOTAL_STEPS}: loss={loss}, accuracy={accuracy}, lr={learning_rate}")
        time.sleep(0.5)

    output_model_id = OUTPUT_MODEL_ID.strip() or f"colab-output-{run['run_id'][:8]}"
    ns.produce_model(output_model_id, role="fine_tuned_model")

except Exception as exc:
    try:
        ns.log(f"Run failed: {type(exc).__name__}: {exc}", level="ERROR")
        ns.finish_run(status="failed")
    finally:
        raise
else:
    ns.finish_run(status="success")
    heartbeat = ns.heartbeat()
    print("Lineage, metrics, and logs sent to NeuralSpace")
    print("Heartbeat:", heartbeat)
